In [1]:
import re
import time
import asyncio
from asyncio import TimeoutError as AsyncTimeoutError
from pathlib import Path
from typing import List, Dict
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy

In [ ]:
FIRECRAWL_API="fc-29599096ac8b426dbf178180c53500ed"
COLUMN_TO_READ_URL_FROM = "G"
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1QtKOB5ChRemg2_wJOxeZhn1qao1ZrRjVK8nExVzbxEI/edit?gid=2011509251#gid=2011509251" 
COLUMN_TO_WRITE_URL_TO = {
	"ABOUT_US": "M",
	"EBOOK": "N",
	"COURSES": "O",
	"RECENT_BLOG": "P",
	"TESTIMONIALS": "Q",
	"WEBINAR": "R",
	"SERVICES": "S",
	"PODCAST": "T",
	"SHOP": "U"
}
COLUMN_TO_PROCESS = "EBOOK" 
CATEGORY_KEYWORDS = {
	"ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
	"EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
	"COURSES": ["course", "training", "academy", "learning", "bootcamp"],
	"RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
	"TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
	"WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
	"SERVICES": ["service", "solutions", "offerings", "capabilities"],
	"PODCAST": ["podcast", "episodes", "listen", "audio"],
	"SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
}
def get_url_filter_for_category(category):
	keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
	return lambda url: any(k in url.lower() for k in keywords)

In [3]:
class Config:
	"""Configuration settings for scraping."""
	max_retries: int = 3
	request_timeout: int = 30
	delay_between_requests: float = 2.0
	max_content_length: int = 10000

# Initialize config
config = Config()

In [4]:
class GoogleSheetsManager:
	"""Manages interactions with Google Sheets."""
	
	def __init__(self, credentials_file: str):
		self.credentials_file = credentials_file
		self.service = self._setup_service()
	
	def _setup_service(self):
		"""Initialize Google Sheets API service."""
		if not Path(self.credentials_file).exists():
			raise FileNotFoundError(f"Credentials file not found: {self.credentials_file}")
		
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		creds = service_account.Credentials.from_service_account_file(
			self.credentials_file, scopes=scopes
		)
		return build('sheets', 'v4', credentials=creds)
	
	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		"""Extract spreadsheet ID from Google Sheets URL."""
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			return match.group(1)
		raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")
	
	def get_urls(self, spreadsheet_id: str) -> List[str]:
		range_name = f"{COLUMN_TO_READ_URL_FROM}2:{COLUMN_TO_READ_URL_FROM}"
		start_row = 2
		try:
			result = self.service.spreadsheets().values().get(
				spreadsheetId=spreadsheet_id,
				range=range_name
			).execute()
			values = result.get('values', [])
			urls_with_rows = [(start_row + i, row[0]) for i, row in enumerate(values) if row and row[0].strip()]
			print(f"Found {len(urls_with_rows)} URLs to scrape")
			return urls_with_rows
		except Exception as e:
			print(f"Error fetching URLs: {e}")
			return []

	
	def update_results(self, spreadsheet_id: str, results: List[Dict]):
		if not results:
			return

		column_letter = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS)
		if not column_letter:
			print(f"Invalid COLUMN_TO_PROCESS: {COLUMN_TO_PROCESS}")
			return

		# Prepare batch update body
		data = [
			{
				"range": f"{column_letter}{result['row']}",
				"values": [[result["content"]]]
			}
			for result in results
		]

		body = {
			"valueInputOption": "RAW",
			"data": data
		}

		try:
			self.service.spreadsheets().values().batchUpdate(
				spreadsheetId=spreadsheet_id,
				body=body
			).execute()
			print(f"✅ Updated {len(data)} rows in column {column_letter}")
		except Exception as e:
			print(f"❌ Failed to update sheet: {e}")


In [5]:
def extract_main_html_content(html: str) -> str:
	soup = BeautifulSoup(html, "html.parser")

	# Try main content tags first
	main = soup.find("main")
	if main:
		return main.get_text(separator="\n", strip=True)

	article = soup.find("article")
	if article:
		return article.get_text(separator="\n", strip=True)

	# Fallback: find largest <div> not known to be junk
	candidates = []
	for div in soup.find_all("div"):
		class_names = " ".join(div.get("class", []))
		if any(x in class_names.lower() for x in ["nav", "footer", "header", "cookie", "modal", "popup"]):
			continue
		text_len = len(div.get_text(strip=True))
		if text_len > 200:  # heuristic: only large divs
			candidates.append((text_len, div))

	if candidates:
		# Return text from the largest good div
		return max(candidates, key=lambda x: x[0])[1].get_text(separator="\n", strip=True)

	# Last resort: return full page text (minus scripts/styles)
	for tag in soup(["script", "style", "noscript"]):
		tag.decompose()
	return soup.get_text(separator="\n", strip=True)

In [6]:
class CrawlThemeExtractor:
	def __init__(self, max_depth=1):
		self.max_depth = max_depth

	def _get_keywords(self, category: str):
		return self.CATEGORY_KEYWORDS.get(category.upper(), [])

	def _extract_summary(self, markdown: str, lines=3):
		all_lines = markdown.splitlines()
		non_empty = [l.strip() for l in all_lines if l.strip()]
		return "\n".join(non_empty[:lines])

	async def extract_thematic_pages(self, main_url: str, category: str):
		keywords = self._get_keywords(category)
		if not keywords:
			print(f"No keywords defined for category '{category}'")
			return

		config = CrawlerRunConfig(
			deep_crawl_strategy=BFSDeepCrawlStrategy(
				max_depth=self.max_depth,
				include_external=True
			),
			verbose=False
		)

		print(f"Crawling {main_url} for category: {category} ({keywords})")
		async with AsyncWebCrawler() as crawler:
			results = await crawler.arun(main_url, config=config)

		filtered = [
			r for r in results if any(k in r.url.lower() for k in keywords)
		]

		if not filtered:
			print("No relevant subpages found.")
			return

		print(f"Found {len(filtered)} matching subpage(s):\n")
		for r in filtered:
			print(f"🔗 {r.url}")
			if r.html:
				clean_text = extract_main_html_content(r.html)
				print(f"{clean_text[:1000]}")
			else:
				print("No HTML content available.")
			print("-" * 50)

async def crawl_single_url(row_num, main_url, extractor, category):
	print(f"\n🌐 Processing row {row_num}: {main_url}")
	try:
		config = CrawlerRunConfig(
			deep_crawl_strategy=BFSDeepCrawlStrategy(max_depth=1, include_external=True),
			verbose=False
		)

		async with AsyncWebCrawler() as crawler:
			results = await asyncio.wait_for(crawler.arun(main_url, config=config), timeout=10)

		# ✅ Log all discovered sub-URLs
		print(f"🔎 Found {len(results)} sub-URLs:")
		for r in results:
			print(f"  - {r.url}")

		# ✅ Apply filter by category
		filtered = [r for r in results if get_url_filter_for_category(category)(r.url)]

		# ✅ Log filtered URLs
		print(f"\n✅ Filtered {len(filtered)} matching URLs for category '{category}':")
		for r in filtered:
			print(f"  - {r.url}")

		if filtered:
			r = filtered[0]
			if r.html:
				content = extract_main_html_content(r.html)
			else:
				content = "⚠️ No HTML content available"
		else:
			content = "⚠️ No relevant pages found"

	except AsyncTimeoutError:
		content = "⏱️ Timed out"
	except Exception as e:
		content = f"❌ Error: {str(e)[:80]}"

	# ✅ Return full content, not summary
	return {"row": row_num, "content": content}

In [7]:
async def process_all_rows_batching(batch_size=5):
	reader = GoogleSheetsManager(CREDENTIALS_FILE)
	spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
	urls_with_rows = reader.get_urls(spreadsheet_id)

	if not urls_with_rows:
		print("❌ No URLs found.")
		return

	category = COLUMN_TO_PROCESS
	extractor = CrawlThemeExtractor(max_depth=1)

	batch = []

	for i, (row_num, url) in enumerate(urls_with_rows, 1):
		print(f"🌐 Processing row {row_num}: {url}")
		try:
			result = await crawl_single_url(row_num, url, extractor, category)
			if result:
				batch.append(result)
		except Exception as e:
			print(f"❌ Failed on row {row_num}: {e}")

		# Flush batch every `batch_size` rows
		if len(batch) >= batch_size:
			reader.update_results(spreadsheet_id, batch)
			batch.clear()

	# Final flush for any leftover results
	if batch:
		reader.update_results(spreadsheet_id, batch)

await process_all_rows_batching()

Found 97 URLs to scrape
🌐 Processing row 3: https://redkiteproject.com

🌐 Processing row 3: https://redkiteproject.com


[INIT].... → Crawl4AI 0.6.3 

🔎 Found 13 sub-URLs:
  - https://redkiteproject.com
  - https://www.redkiteproject.com
  - https://www.facebook.com/search/top?q=red+kite+project
  - https://www.linkedin.com/company/red-kite-project
  - https://twitter.com/RedKiteProject
  - https://rise.articulate.com/share/JWdf_671A7wWJ_JywJQk9978hdOonBZw
  - https://www.redkiteproject.com/post/leading-change-and-transforming-conflict
  - https://www.redkiteproject.com/about
  - https://www.redkiteproject.com/news-updates
  - https://www.redkiteproject.com/clients
  - https://www.redkiteproject.com/post/resolve-conflict-effectively-talk
  - https://www.redkiteproject.com/post/what-does-a-15th-century-blackmailer-have-to-teach-us-about-emotional-intelligence
  - https://www.redkiteproject.com/services

✅ Filtered 1 matching URLs for category 'RECENT_BLOG':
  - https://www.redkiteproject.com/news-updates
🌐 Processing row 4: https://lrscpa.com

🌐 Processing row 4: https://lrscpa.com


[INIT].... → Crawl4AI 0.6.3 

Invalid URL: tel:7325318000, error: Missing scheme or netloc
Invalid URL: mailto:SalSchibell@lrscpa.com, error: Missing scheme or netloc
Invalid URL: mailto:salschibell@lrscpa.com, error: Missing scheme or netloc
Invalid URL: tel:7325318000,225, error: Missing scheme or netloc


...

✅ Updated 5 rows in column P
🌐 Processing row 98: https://microburstlearning.com

🌐 Processing row 98: https://microburstlearning.com


[INIT].... → Crawl4AI 0.6.3 

🔎 Found 1 sub-URLs:
  - https://microburstlearning.com

✅ Filtered 0 matching URLs for category 'RECENT_BLOG':
🌐 Processing row 99: https://tcgraleigh.com

🌐 Processing row 99: https://tcgraleigh.com


[INIT].... → Crawl4AI 0.6.3 

🔎 Found 1 sub-URLs:
  - https://tcgraleigh.com

✅ Filtered 0 matching URLs for category 'RECENT_BLOG':
✅ Updated 2 rows in column P


In [8]:
# async def process_all_rows():
# 	# Step 1: Setup
# 	reader = GoogleSheetsManager(CREDENTIALS_FILE)
# 	spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
# 	urls_with_rows = reader.get_urls(spreadsheet_id)

# 	if not urls_with_rows:
# 		print("❌ No URLs found.")
# 		return

# 	category = COLUMN_TO_PROCESS
# 	extractor = CrawlThemeExtractor(max_depth=1)

# 	# Step 2: Process each row sequentially
# 	results_to_update = []
# 	for row_num, url in urls_with_rows:
# 		print(f"🌐 Processing row {row_num}: {url}")
# 		try:
# 			result = await crawl_single_url(row_num, url, extractor, category)
# 			if result:
# 				results_to_update.append(result)
# 		except Exception as e:
# 			print(f"❌ Failed on row {row_num}: {e}")

# 	# Step 3: Write results to sheet
# 	if results_to_update:
# 		reader.update_results(spreadsheet_id, results_to_update)
# 	else:
# 		print("⚠️ No results to update.")

# await process_all_rows()


In [9]:
# import asyncio
# from asyncio import TimeoutError as AsyncTimeoutError

# async def process_one_row_sample():
# 	reader = GoogleSheetsManager(CREDENTIALS_FILE)
# 	spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
# 	urls_with_rows = reader.get_urls(spreadsheet_id)

# 	if not urls_with_rows:
# 		print("❌ No URLs found in sheet.")
# 		return

# 	# Take the first valid URL
# 	row_num, main_url = urls_with_rows[1]
# 	print(f"🌐 Processing sample row {row_num}: {main_url}")

# 	category = COLUMN_TO_PROCESS
# 	extractor = CrawlThemeExtractor(max_depth=1)

# 	try:
# 		config = CrawlerRunConfig(
# 			deep_crawl_strategy=BFSDeepCrawlStrategy(max_depth=1, include_external=False),
# 			verbose=False
# 		)

# 		async with AsyncWebCrawler() as crawler:
# 			results = await asyncio.wait_for(crawler.arun(main_url, config=config), timeout=10)

# 		filtered = [r for r in results if get_url_filter_for_category(category)(r.url)]

# 		if filtered:
# 			r = filtered[0]
# 			if r.html:
# 				content = extract_main_html_content(r.html)
# 				summary = extractor._extract_summary(content, lines=5)
# 			else:
# 				summary = "⚠️ No HTML content available"
# 		else:
# 			summary = "⚠️ No relevant pages found"

# 	except AsyncTimeoutError:
# 		summary = "⏱️ Timed out"
# 	except Exception as e:
# 		summary = f"❌ Error: {str(e)[:80]}"

# 	# Update back to the same row
# 	column_letter = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS)
# 	if not column_letter:
# 		print(f"❌ Invalid column mapping for {COLUMN_TO_PROCESS}")
# 		return

# 	range_str = f"{column_letter}{row_num}"
# 	print(f"📤 Writing result to sheet at {range_str}")
	
# 	try:
# 		reader.service.spreadsheets().values().update(
# 			spreadsheetId=spreadsheet_id,
# 			range=range_str,
# 			valueInputOption='RAW',
# 			body={'values': [[summary]]}
# 		).execute()
# 		print("✅ Sheet updated successfully.")
# 	except Exception as e:
# 		print(f"❌ Failed to update sheet: {e}")

# # Run in notebook
# await process_one_row_sample()

In [10]:
# async def run_thematic_crawl_and_update_sheet():
# 	reader = GoogleSheetsManager(CREDENTIALS_FILE)
# 	spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
# 	urls_with_rows = reader.get_urls(spreadsheet_id)

# 	category = COLUMN_TO_PROCESS
# 	extractor = CrawlThemeExtractor(max_depth=1)

# 	# Concurrent crawling
# 	tasks = [
# 		crawl_single_url(row_num, url, extractor, category)
# 		for row_num, url in urls_with_rows
# 	]
# 	results_for_update = await asyncio.gather(*tasks)

# 	# Format results for update
# 	column_letter = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS)
# 	if not column_letter:
# 		print(f"❌ Invalid column for processing: {COLUMN_TO_PROCESS}")
# 		return

# 	max_row = max(result['row'] for result in results_for_update)
# 	content_data = [[""] for _ in range(max_row + 1)]
# 	for result in results_for_update:
# 		content_data[result['row']] = [result['content']]
# 	content_data = content_data[2:]

# 	# Update Google Sheet
# 	try:
# 		reader.service.spreadsheets().values().update(
# 			spreadsheetId=spreadsheet_id,
# 			range=f"{column_letter}2:{column_letter}{2 + len(content_data) - 1}",
# 			valueInputOption='RAW',
# 			body={'values': content_data}
# 		).execute()
# 		print(f"✅ Successfully updated {len(results_for_update)} rows in column {column_letter}")
# 	except Exception as e:
# 		print(f"❌ Failed to update Google Sheet: {e}")

# # Run the orchestration
# await run_thematic_crawl_and_update_sheet()